# 18 — Initial Ward25 Analytical Scoring

This notebook creates a first-pass, configurable scoring layer from the target model input table.

The output is a **review/watchlist layer**, not a final operational decision list.

Default scope: **North West**

The notebook can also produce England/Wales-wide and region-level outputs once you are ready to broaden the analysis.

## 18.1 Project paths and switches

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import json

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name.lower() == "notebooks":
    PROJECT_DIR = NOTEBOOK_DIR.parent
else:
    PROJECT_DIR = NOTEBOOK_DIR

DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"

MODEL_DIR = PROCESSED_DIR / "target_model_v1"
MODEL_INPUT_DIR = MODEL_DIR / "inputs"
MODEL_OUTPUT_DIR = MODEL_DIR / "outputs"
MODEL_REVIEW_DIR = MODEL_DIR / "review"

for d in [MODEL_OUTPUT_DIR, MODEL_REVIEW_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Scope controls.
DEFAULT_SCOPE_COLUMN = "scope_north_west"
DEFAULT_SCOPE_NAME = "north_west"

GENERATE_ALL_AVAILABLE_OUTPUTS = True
GENERATE_REGION_OUTPUTS = True  # Set True if you want per-region output files immediately.

# Score weights. These are transparent starting weights and should be sensitivity-tested later.
SCORE_WEIGHTS = {
    "demographic_relevance_score": 0.35,
    "electoral_opportunity_score": 0.30,
    "political_openness_score": 0.25,
    "data_confidence_score": 0.10,
}

print("Project:", PROJECT_DIR)
print("Model input dir:", MODEL_INPUT_DIR)
print("Model output dir:", MODEL_OUTPUT_DIR)

Project: c:\Users\keena\Documents\Electoral_Tribes
Model input dir: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v1\inputs
Model output dir: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v1\outputs


## 18.2 Load model input

In [2]:
INPUT_PATH = MODEL_INPUT_DIR / "target_model_input_ward25_v1.csv"
NW_INPUT_PATH = MODEL_INPUT_DIR / "target_model_input_north_west_ward25_v1.csv"

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Missing model input: {INPUT_PATH}. Run Notebook 17 first.")

df = pd.read_csv(INPUT_PATH, low_memory=False)

print("Rows:", len(df))
print("Columns:", len(df.columns))
display(df.head())

Rows: 7572
Columns: 119


,LAD25CD,LAD25NM,WD25CD,WD25NM,RGN25CD,RGN25NM,analysis_region,country_inferred,scope_north_west,scope_england,scope_wales,scope_all_available,population,oa_count,dominant_cluster,dominant_cluster_name,second_cluster,second_cluster_name,dominant_cluster_share,second_cluster_share,cluster_fragmentation_index,is_mixed_ward,is_clear_dominant_ward,is_highly_fragmented,cluster_0_share,cluster_1_share,cluster_2_share,cluster_3_share,cluster_4_share,cluster_5_share,cluster_6_share,student_transient_youth_share,rooted_older_homeowners_share,stable_suburban_professionals_share,cosmopolitan_young_professional_core_share,settled_working_families_skilled_trades_suburbs_share,settled_diverse_urban_communities_share,post_industrial_estates_deprived_working_communities_share,age_0_14_pct,age_15_24_pct,age_25_34_pct,age_35_49_pct,age_50_64_pct,age_65_plus_pct,uk_born_pct,non_uk_born_pct,resident_10_plus_years_pct,resident_less_5_years_pct,white_british_pct,white_other_pct,non_white_pct,owned_pct,owns_outright_pct,owns_mortgage_pct,social_rented_pct,private_rented_pct,house_type_pct,flat_type_pct,managerial_professional_pct,skilled_traditional_pct,routine_service_elementary_pct,employed_pct,unemployed_pct,full_time_student_pct,retired_pct,long_term_sick_disabled_pct,no_qualifications_pct,level_1_2_pct,apprenticeship_pct,level_4_plus_pct,one_person_household_pct,married_couple_family_pct,lone_parent_family_pct,latest_election_WD25NM,latest_election_LAD25CD,latest_election_LAD25NM,latest_election_source_year,latest_election_allocated_electorate,latest_election_allocated_valid_votes,latest_election_allocated_ballots,latest_election_allocated_invalid_votes,latest_election_allocated_top_party_votes,latest_election_allocated_runner_up_party_votes,latest_election_allocated_con_votes,latest_election_allocated_lab_votes,latest_election_allocated_ld_votes,latest_election_allocated_green_votes,latest_election_allocated_reform_ukip_brexit_votes,latest_election_allocated_independent_votes,latest_election_allocated_sdp_votes,latest_election_allocated_other_votes,latest_election_contributing_oa_rows,latest_election_contributing_result_areas,latest_election_contributing_source_years,latest_election_con_share,latest_election_lab_share,latest_election_ld_share,latest_election_green_share,latest_election_reform_ukip_brexit_share,latest_election_independent_share,latest_election_sdp_share,latest_election_other_share,latest_election_top_party_bucket,latest_election_runner_up_party_bucket,latest_election_top_party_votes_allocated,latest_election_runner_up_party_votes_allocated,latest_election_margin_votes_allocated,latest_election_margin_pct_allocated,latest_election_party_fragmentation_index,latest_election_effective_number_of_parties,latest_election_aggregation_label,latest_election_latest_layer_note,has_latest_election_layer,has_valid_vote_data,has_margin_data,boundary_caveat,county_election_caveat,target_model_ready,data_confidence_note
0,E06000001,Hartlepool,E05013038,Burn Valley,E12000001,North East,North East,England,False,True,False,True,7633,26,6.0,Post-Industrial Estates / Deprived Working Com...,2.0,Stable Suburban Professionals,0.458404,0.191406,0.703925,False,False,True,0.043495,0.167169,0.191406,0.0,0.139526,0.0,0.458404,0.043495,0.167169,0.191406,0.0,0.139526,0.0,0.458404,0.174112,0.137037,0.106511,0.179353,0.206996,0.194157,0.948513,0.051487,0.029208,0.014931,0.925465,0.018863,0.052135,0.554228,0.304751,0.249477,0.217807,0.224081,0.843032,0.155774,0.371084,0.207229,0.333219,0.467880,0.046208,0.092256,0.234262,0.084688,0.225000,0.240806,0.062903,0.267419,0.389420,0.251046,0.124626,Burn Valley,E06000001,Hartlepool,2024.0,5818.0,1686.0,0.0,0.0,919.0,339.0,339.0,919.0,0.0,0.0,270.0,158.0,0.0,0.0,26.0,1.0,2024.0,0.201068,0.545077,0.0,0.0,0.160142,0.093713,0.0,0.000000,lab,con,919.0,339.0,580.0,0.344009,0.628035,2.688426,ward25_by_source_year,latest_available_source_year_after_oa21_crosswalk,True,True,True,NaN,NaN,True,No major caveat.
1,E06000001,Hartlep

## 18.3 Helper functions

Scores are scaled 0–100. Higher scores indicate stronger fit to that component's rule.

In [3]:
def to_num(s):
    return pd.to_numeric(s, errors="coerce")


def minmax_score(series, higher_is_better=True, fill_value=0):
    s = to_num(series).replace([np.inf, -np.inf], np.nan)

    if s.notna().sum() == 0:
        return pd.Series(fill_value, index=series.index, dtype=float)

    lo = s.quantile(0.01)
    hi = s.quantile(0.99)

    if pd.isna(lo) or pd.isna(hi) or hi == lo:
        out = pd.Series(50, index=series.index, dtype=float)
    else:
        clipped = s.clip(lo, hi)
        out = (clipped - lo) / (hi - lo) * 100

    if not higher_is_better:
        out = 100 - out

    return out.fillna(fill_value).clip(0, 100)


def weighted_average(df, cols_weights):
    total_weight = sum(w for c, w in cols_weights if c in df.columns)
    if total_weight == 0:
        return pd.Series(0, index=df.index, dtype=float)

    out = pd.Series(0, index=df.index, dtype=float)

    for col, weight in cols_weights:
        if col in df.columns:
            out += to_num(df[col]).fillna(0) * weight

    return out / total_weight


def safe_share(col):
    if col in df.columns:
        return to_num(df[col]).fillna(0).clip(lower=0)
    return pd.Series(0, index=df.index, dtype=float)


def save_scope_output(data, scope_name, filename_stub):
    safe = re.sub(r"[^a-zA-Z0-9]+", "_", scope_name.lower()).strip("_")
    out_path = MODEL_OUTPUT_DIR / f"{filename_stub}_{safe}_v1.csv"
    data.to_csv(out_path, index=False)
    print("Saved:", out_path)
    return out_path

## 18.4 Build score components

Component meanings:

- **Demographic relevance**: composition of selected K7 cluster shares
- **Electoral opportunity**: closeness/fragmentation/scale of latest result
- **Political openness**: evidence of non-dominant-party or fragmented vote structure
- **Data confidence**: quality, recency and caveat flags

These are analytical indicators. The weights can and should be reviewed later.

In [4]:
scored = df.copy()

# -------------------------
# Demographic relevance
# -------------------------
demo_cluster_weights = [
    ("post_industrial_estates_deprived_working_communities_share", 0.45),
    ("settled_working_families_skilled_trades_suburbs_share", 0.35),
    ("rooted_older_homeowners_share", 0.20),
]

scored["demographic_relevance_raw"] = weighted_average(scored, demo_cluster_weights) * 100
scored["demographic_relevance_score"] = scored["demographic_relevance_raw"].clip(0, 100)

# -------------------------
# Electoral opportunity
# -------------------------
# Lower margin and lower top-party dominance score higher.
if "latest_election_margin_pct_allocated" in scored.columns:
    scored["margin_competitiveness_score"] = minmax_score(
        scored["latest_election_margin_pct_allocated"].abs(),
        higher_is_better=False,
        fill_value=0
    )
else:
    scored["margin_competitiveness_score"] = 0

if "latest_election_top_party_votes_allocated" in scored.columns and "latest_election_allocated_valid_votes" in scored.columns:
    top_party_share = to_num(scored["latest_election_top_party_votes_allocated"]) / to_num(scored["latest_election_allocated_valid_votes"])
    scored["top_party_dominance_inverse_score"] = minmax_score(top_party_share, higher_is_better=False, fill_value=0)
else:
    scored["top_party_dominance_inverse_score"] = 0

if "latest_election_allocated_valid_votes" in scored.columns:
    scored["valid_vote_threshold_score"] = minmax_score(
        scored["latest_election_allocated_valid_votes"],
        higher_is_better=False,
        fill_value=0
    )
else:
    scored["valid_vote_threshold_score"] = 0

if "latest_election_party_fragmentation_index" in scored.columns:
    scored["electoral_fragmentation_score"] = minmax_score(
        scored["latest_election_party_fragmentation_index"],
        higher_is_better=True,
        fill_value=0
    )
else:
    scored["electoral_fragmentation_score"] = 0

scored["electoral_opportunity_score"] = (
    scored["margin_competitiveness_score"] * 0.40
    + scored["top_party_dominance_inverse_score"] * 0.25
    + scored["valid_vote_threshold_score"] * 0.20
    + scored["electoral_fragmentation_score"] * 0.15
).clip(0, 100)

# -------------------------
# Political openness
# -------------------------
# This is a structural indicator: it measures fragmented / non-main-party / localist vote patterns.
reform_share = safe_share("latest_election_reform_ukip_brexit_share")
ind_share = safe_share("latest_election_independent_share")
other_share = safe_share("latest_election_other_share")
green_share = safe_share("latest_election_green_share")
ld_share = safe_share("latest_election_ld_share")
con_share = safe_share("latest_election_con_share")
lab_share = safe_share("latest_election_lab_share")

scored["non_main_party_share"] = (reform_share + ind_share + other_share + green_share + ld_share).clip(0, 1)
scored["lab_con_combined_share"] = (lab_share + con_share).clip(0, 1)
scored["lab_con_inverse_score"] = (1 - scored["lab_con_combined_share"]).clip(0, 1) * 100

scored["non_main_party_score"] = minmax_score(scored["non_main_party_share"], higher_is_better=True, fill_value=0)

if "latest_election_effective_number_of_parties" in scored.columns:
    scored["effective_parties_score"] = minmax_score(scored["latest_election_effective_number_of_parties"], higher_is_better=True, fill_value=0)
else:
    scored["effective_parties_score"] = 0

scored["political_openness_score"] = (
    scored["non_main_party_score"] * 0.40
    + scored["electoral_fragmentation_score"] * 0.25
    + scored["effective_parties_score"] * 0.20
    + scored["lab_con_inverse_score"] * 0.15
).clip(0, 100)

# -------------------------
# Data confidence
# -------------------------
scored["data_confidence_score"] = 100.0

for col in ["has_latest_election_layer", "has_valid_vote_data", "has_margin_data", "target_model_ready"]:
    if col in scored.columns:
        scored[col] = scored[col].astype(str).str.lower().isin(["true", "1", "yes", "y"])
    else:
        scored[col] = False

scored.loc[~scored["has_latest_election_layer"], "data_confidence_score"] -= 40
scored.loc[~scored["has_valid_vote_data"], "data_confidence_score"] -= 30
scored.loc[~scored["has_margin_data"], "data_confidence_score"] -= 20

if "latest_election_source_year" in scored.columns:
    scored["latest_election_source_year"] = to_num(scored["latest_election_source_year"])
    scored.loc[scored["latest_election_source_year"].lt(2023), "data_confidence_score"] -= 10

if "boundary_caveat" in scored.columns:
    scored.loc[scored["boundary_caveat"].fillna("").ne(""), "data_confidence_score"] -= 10

if "county_election_caveat" in scored.columns:
    scored.loc[scored["county_election_caveat"].fillna("").ne(""), "data_confidence_score"] -= 10

scored["data_confidence_score"] = scored["data_confidence_score"].clip(0, 100)

# -------------------------
# Combined watchlist score
# -------------------------
scored["initial_watchlist_score"] = (
    scored["demographic_relevance_score"] * SCORE_WEIGHTS["demographic_relevance_score"]
    + scored["electoral_opportunity_score"] * SCORE_WEIGHTS["electoral_opportunity_score"]
    + scored["political_openness_score"] * SCORE_WEIGHTS["political_openness_score"]
    + scored["data_confidence_score"] * SCORE_WEIGHTS["data_confidence_score"]
).clip(0, 100)

display(
    scored[[
        "initial_watchlist_score",
        "demographic_relevance_score",
        "electoral_opportunity_score",
        "political_openness_score",
        "data_confidence_score",
    ]].describe()
)

,initial_watchlist_score,demographic_relevance_score,electoral_opportunity_score,political_openness_score,data_confidence_score
count,7572.000000,7572.000000,7572.000000,7572.000000,7572.000000
mean,44.854242,19.590736,61.458139,39.790094,96.125198
std,8.950222,11.586772,16.444051,14.550625,10.652428
min,4.750000,0.000000,0.000000,0.000000,10.000000
25%,39.198208,10.374522,51.666852,29.486668,90.000000
50%,45.939812,20.809202,64.994576,42.263599,100.000000
75%,51.420000,28.392623,73.995443,50.610122,100.000000
max,65.894989,45.000000,90.520433,72.907941,100.000000


## 18.5 Assign review bands

The bands are percentile-based within the currently loaded full dataset. They should be treated as review categories, not decisions.

In [5]:
# Percentile rank: higher = stronger on this model's combined indicator.
scored["initial_watchlist_percentile"] = scored["initial_watchlist_score"].rank(pct=True) * 100

scored["review_band"] = pd.cut(
    scored["initial_watchlist_percentile"],
    bins=[0, 50, 75, 90, 100],
    labels=["Review D", "Review C", "Review B", "Review A"],
    include_lowest=True
).astype(str)

scored.loc[~scored["target_model_ready"], "review_band"] = "Manual / incomplete data"

display(scored["review_band"].value_counts(dropna=False))

review_band
Review D                    3631
Review C                    1893
Review B                    1135
Review A                     758
Manual / incomplete data     155
Name: count, dtype: int64

## 18.6 Save national/all-available scored file

In [6]:
score_path = MODEL_OUTPUT_DIR / "initial_watchlist_scores_ward25_all_available_v1.csv"
scored.to_csv(score_path, index=False)

component_cols = [
    "LAD25CD", "LAD25NM", "WD25CD", "WD25NM", "analysis_region",
    "dominant_cluster_name", "second_cluster_name",
    "initial_watchlist_score", "initial_watchlist_percentile", "review_band",
    "demographic_relevance_score", "electoral_opportunity_score", "political_openness_score", "data_confidence_score",
    "margin_competitiveness_score", "top_party_dominance_inverse_score", "valid_vote_threshold_score",
    "electoral_fragmentation_score", "non_main_party_score", "effective_parties_score", "lab_con_inverse_score",
    "data_confidence_note", "boundary_caveat", "county_election_caveat",
]

component_cols = [c for c in component_cols if c in scored.columns]

components = scored[component_cols].copy()
components.to_csv(MODEL_OUTPUT_DIR / "target_score_components_ward25_all_available_v1.csv", index=False)

print("Saved:", score_path)
print("Saved score components")

Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v1\outputs\initial_watchlist_scores_ward25_all_available_v1.csv
Saved score components


## 18.7 Create scoped review outputs

Default is North West. Optional all-region generation can be turned on in section 18.1.

In [7]:
def make_scope_outputs(data, scope_name):
    scope_dir = MODEL_OUTPUT_DIR / "scopes" / re.sub(r"[^a-zA-Z0-9]+", "_", scope_name.lower()).strip("_")
    scope_dir.mkdir(parents=True, exist_ok=True)

    data = data.copy()

    # Main scored file.
    data.sort_values("initial_watchlist_score", ascending=False).to_csv(
        scope_dir / f"initial_watchlist_scores_{scope_name}_v1.csv",
        index=False
    )

    # Component review files.
    review_cols = [c for c in component_cols if c in data.columns]

    data.sort_values("demographic_relevance_score", ascending=False)[review_cols].head(50).to_csv(
        scope_dir / f"top_50_demographic_relevance_{scope_name}_v1.csv",
        index=False
    )

    data.sort_values("electoral_opportunity_score", ascending=False)[review_cols].head(50).to_csv(
        scope_dir / f"top_50_electoral_opportunity_{scope_name}_v1.csv",
        index=False
    )

    data.sort_values("political_openness_score", ascending=False)[review_cols].head(50).to_csv(
        scope_dir / f"top_50_political_openness_{scope_name}_v1.csv",
        index=False
    )

    data.sort_values("initial_watchlist_score", ascending=False)[review_cols].head(100).to_csv(
        scope_dir / f"top_100_initial_watchlist_{scope_name}_v1.csv",
        index=False
    )

    # Council summary.
    council_summary = (
        data.groupby(["LAD25CD", "LAD25NM"], as_index=False)
        .agg(
            wards=("WD25CD", "nunique"),
            mean_initial_watchlist_score=("initial_watchlist_score", "mean"),
            median_initial_watchlist_score=("initial_watchlist_score", "median"),
            review_a_count=("review_band", lambda s: (s == "Review A").sum()),
            review_b_count=("review_band", lambda s: (s == "Review B").sum()),
            ready_wards=("target_model_ready", "sum"),
            mean_demographic_relevance_score=("demographic_relevance_score", "mean"),
            mean_electoral_opportunity_score=("electoral_opportunity_score", "mean"),
            mean_political_openness_score=("political_openness_score", "mean"),
            mean_data_confidence_score=("data_confidence_score", "mean"),
        )
        .sort_values(["mean_initial_watchlist_score"], ascending=False)
    )

    council_summary.to_csv(
        scope_dir / f"council_review_summary_{scope_name}_v1.csv",
        index=False
    )

    print(f"Saved scope outputs for {scope_name}: {scope_dir}")
    return scope_dir


# Default North West outputs.
nw = scored[scored[DEFAULT_SCOPE_COLUMN]].copy()
make_scope_outputs(nw, DEFAULT_SCOPE_NAME)

if GENERATE_ALL_AVAILABLE_OUTPUTS:
    make_scope_outputs(scored, "all_available")

if GENERATE_REGION_OUTPUTS and "analysis_region" in scored.columns:
    for region_name, region_df in scored.groupby("analysis_region", dropna=False):
        safe_region = re.sub(r"[^a-zA-Z0-9]+", "_", str(region_name).lower()).strip("_")
        if safe_region and len(region_df) > 0:
            make_scope_outputs(region_df, safe_region)

Saved scope outputs for north_west: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v1\outputs\scopes\north_west
Saved scope outputs for all_available: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v1\outputs\scopes\all_available
Saved scope outputs for east_midlands: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v1\outputs\scopes\east_midlands
Saved scope outputs for east_of_england: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v1\outputs\scopes\east_of_england
Saved scope outputs for london: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v1\outputs\scopes\london
Saved scope outputs for north_east: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v1\outputs\scopes\north_east
Saved scope outputs for north_west: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v1\outputs\scopes\north_west
Saved scope outputs for south_east: c:\Us

## 18.8 Score QA summary

Use this to check whether a component is dominating the combined score.

In [8]:
score_cols = [
    "initial_watchlist_score",
    "demographic_relevance_score",
    "electoral_opportunity_score",
    "political_openness_score",
    "data_confidence_score",
]

corr = scored[score_cols].corr()
corr.to_csv(MODEL_REVIEW_DIR / "initial_score_component_correlation_v1.csv")

display(corr)

qa_summary = scored.groupby("analysis_region", dropna=False).agg(
    wards=("WD25CD", "nunique"),
    mean_score=("initial_watchlist_score", "mean"),
    median_score=("initial_watchlist_score", "median"),
    ready_wards=("target_model_ready", "sum"),
    review_a=("review_band", lambda s: (s == "Review A").sum()),
    review_b=("review_band", lambda s: (s == "Review B").sum()),
).reset_index()

qa_summary.to_csv(MODEL_REVIEW_DIR / "initial_score_summary_by_region_v1.csv", index=False)
display(qa_summary)

,initial_watchlist_score,demographic_relevance_score,electoral_opportunity_score,political_openness_score,data_confidence_score
initial_watchlist_score,1.000000,0.498083,0.805730,0.683883,0.439091
demographic_relevance_score,0.498083,1.000000,0.055731,0.027697,0.025253
electoral_opportunity_score,0.805730,0.055731,1.000000,0.440400,0.422646
political_openness_score,0.683883,0.027697,0.440400,1.000000,0.186193
data_confidence_score,0.439091,0.025253,0.422646,0.186193,1.000000


,analysis_region,wards,mean_score,median_score,ready_wards,review_a,review_b
0,East Midlands,761,48.721306,50.137388,746,116,189
1,East of England,947,46.277751,46.625667,938,95,142
2,London,689,32.626960,33.018279,677,0,2
3,North East,334,48.010457,48.996617,334,44,66
4,North West,825,43.407355,43.948770,824,36,81
5,South East,1269,45.253787,46.182163,1229,104,182
6,South West,821,49.922953,50.587358,817,196,187
7,Wales,762,43.494732,43.976223,690,42,108
8,West Midlands,754,46.044077,47.810908,753,98,137
9,Yorkshire and The Humber,410,44.228837,44.433346,409,27,41
